# Data cleaning and preparations
Download `TV_kiirustabloo`. Using filtering on the website, download 1_idaringtee from `UUS_kiirustablood` by years 2023, 2024, 2025 separately and name the files `1_idaringtee_year.csv`. Download all the rest of the data by radar speed sign (one file per sign) for each sign listed in `TV_kiirustabloo`, selecting only the years 2023, 2024, 2025. Name the files with the name of the speed sign.

**Skip the following signs**
1. 1_riia (2025 missing)
2. 1_arukula
3. 1_kummeli
4. 1_ulikooli
5. 2_riia
6. 1_nurme (2023 missing)
7. 1_kungla
8. 1_pohja_pst
17. ...

So all in all you should have 10 files downloaded.

## TODO
1. [ ] Decide what to do with missing values (e.g. january 1-16 values in 1_tahe)
2. [ ] Decide whether to incorporate 2026 as well. Where and how.
3. [ ] Decide whether to incorporate recently added signs from after 2023.

In [1]:
import glob
import pandas as pd

In [33]:
# function for turning TV_kiirustabloo.csv into a dataframe and removing unnecessary columns
def process_sign_info_file(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename)
    df_sub = df[['X','Y','Tunnus','Maks_kiirus']] # kas tahame koordinaate alles hoida?
    return df_sub

In [46]:
# 4 km/h is always subtracted from any speeding incidents (mõõtemääramatus)
# speed cameras register speedings starting from 7 km/h over the limit, mobile police radars from 10 km/h over
# currently default speeding tolerance is therefore 6 km/h, since it's the lowest you can get away with
# currently speeds under 5 km/h and over 150 km/h are removed, we should assess if these bounds are suitable
# hours where less than 5 cars passed
# weekday -> monday = 0
def process_measured_speeds_file(measured_speeds_file: str, sign_info_file: str, min_passing_cars=5, speeding_tolerance_kmh=6) -> pd.DataFrame:
    df = pd.read_csv(
        measured_speeds_file,
        usecols=["Tunnus", "V1", "Year", "Month", "Day", "Hour"],
        dtype={"V1": "float32", "Tunnus": "category"},
    )

    # eemalda ilmsed mõõtmisvead (kohanda piire oma andmetele)
    df = df[df["V1"].between(5, 150)]

    # lisa piirang ja märgi ületajad
    limits_df = process_sign_info_file(sign_info_file)
    df = df.merge(limits_df, on='Tunnus')
    df["Speeders_count"] = (df["V1"] > df["Maks_kiirus"] + speeding_tolerance_kmh).astype("int8")

    hour_stats_df = (
        df.groupby(["Tunnus", "Year", "Month", "Day", "Hour"], observed=True) # add "X","Y" to groupby to see coordinate columns
          .agg(passers_count=("V1", "size"),
               avg_speed=("V1", "mean"),
               speeders_count=("Speeders_count", "sum"))
          .reset_index()
    )

    hour_stats_df["speeders_share"] = hour_stats_df["speeders_count"] / hour_stats_df["passers_count"]
    hour_stats_df = hour_stats_df[hour_stats_df["passers_count"] >= min_passing_cars]

    # lisa nädalapäev, kasulik hilisemaks mudeliks
    kuupaev = pd.to_datetime(
        hour_stats_df[["Year", "Month", "Day"]].rename(columns=str.lower)
    )
    hour_stats_df["weekday"] = kuupaev.dt.dayofweek  # 0 = esmaspäev

    return hour_stats_df

In [47]:
df = process_measured_speeds_file("data/2_ulikooli.csv","data/TV_kiirustabloo.csv")
df.head(20)

,Tunnus,Year,Month,Day,Hour,passers_count,avg_speed,speeders_count,speeders_share,weekday
0,2_ulikooli,2023,11,16,15,40,25.275000,19,0.475000,3
1,2_ulikooli,2023,11,16,16,184,21.820652,37,0.201087,3
2,2_ulikooli,2023,11,16,17,91,25.494505,39,0.428571,3
3,2_ulikooli,2023,11,16,18,121,22.768595,28,0.231405,3
4,2_ulikooli,2023,11,16,19,169,21.721893,24,0.142012,3
5,2_ulikooli,2023,11,16,20,162,22.716049,36,0.222222,3
6,2_ulikooli,2023,11,16,21,123,23.373983,38,0.308943,3
7,2_ulikooli,2023,11,16,22,154,24.272728,48,0.311688,3
8,2_ulikooli,2023,11,16,23,101,23.722773,25,0.247525,3
9,2_ulikooli,2023,11,17,0,17,31.000000,13,0.764706,4
